# Phase 1 — Real backend server (Kaggle)
Serves the **real** F5-TTS / LivePortrait / MuseTalk pipeline behind the FastAPI backend, exposed via a public cloudflared URL the website can call.

**Settings → Accelerator: GPU T4 x2, Internet: ON.** **Add Data → `ai-avatar-weights`** (mounted at `/kaggle/input/ai-avatar-weights`).

Run cells top to bottom. Env builds take ~30-50 min the first time.

In [ ]:
# 1. Clone (or update) the repo.
import os, subprocess
os.chdir('/kaggle/working')
if os.path.exists('Ai-Avatar'):
    subprocess.run(['git','-C','Ai-Avatar','pull','--ff-only'], check=False)
else:
    subprocess.run(['git','clone','https://github.com/Yashvant203/Ai-Avatar.git'], check=True)
print('repo ready')

In [ ]:
# 2. Inspect the mounted weights dataset, then restore into /tmp/aiavatar.
# Confirm the layout matches restore_weights.sh; adjust SRC if the dataset nests differently.
import subprocess
subprocess.run(['find','/kaggle/input/ai-avatar-weights','-maxdepth','2'], check=False)
subprocess.run(['bash','/kaggle/working/Ai-Avatar/ml_models/runners/restore_weights.sh','/kaggle/input/ai-avatar-weights/aa_cache','/tmp/aiavatar'], check=False)

In [ ]:
# 3. Build envF5 (F5-TTS). Weights restored above, but pip still installs. ~8-12 min.
!bash /kaggle/working/Ai-Avatar/ml_models/runners/setup_env_f5.sh /tmp/aiavatar

In [ ]:
# 4. Build envLP (LivePortrait + insightface). ~8-12 min.
!bash /kaggle/working/Ai-Avatar/ml_models/runners/setup_env_lp.sh /tmp/aiavatar

In [ ]:
# 5. Build envMT (MuseTalk + mmcv). The riskiest install. ~15-25 min.
!bash /kaggle/working/Ai-Avatar/ml_models/runners/setup_env_mt.sh /tmp/aiavatar

In [ ]:
# 6. Install the FastAPI backend into the base env + run DB migrations.
import os, subprocess
os.makedirs('/tmp/aiavatar/storage', exist_ok=True)
subprocess.run(['pip','install','-e','/kaggle/working/Ai-Avatar/backend'], check=True)
subprocess.run(['alembic','upgrade','head'], cwd='/kaggle/working/Ai-Avatar/backend', check=True)
print('backend installed + migrated')

In [ ]:
# 7. Launch the worker + API (PIPELINE_BACKEND=real), both with cwd=backend.
import os, secrets, subprocess, time
srv_env = {**os.environ,
    'PIPELINE_BACKEND': 'real',
    'JWT_SECRET': secrets.token_urlsafe(48),
    'CORS_ORIGINS': '*',
    'STORAGE_DIR': '/tmp/aiavatar/storage',
    'AI_ROOT': '/tmp/aiavatar',
}
wlog = open('/tmp/aiavatar/worker.log','w'); alog = open('/tmp/aiavatar/api.log','w')
worker = subprocess.Popen(['python','-m','worker.main'],
    cwd='/kaggle/working/Ai-Avatar/backend', env=srv_env, stdout=wlog, stderr=subprocess.STDOUT)
api = subprocess.Popen(['uvicorn','app.main:app','--host','0.0.0.0','--port','7860'],
    cwd='/kaggle/working/Ai-Avatar/backend', env=srv_env, stdout=alog, stderr=subprocess.STDOUT)
time.sleep(8)
import urllib.request
try:
    print('health:', urllib.request.urlopen('http://127.0.0.1:7860/healthcheck', timeout=5).read())
except Exception as e:
    print('health check failed:', e); print(open('/tmp/aiavatar/api.log').read()[-2000:])

In [ ]:
# 8. Public URL via cloudflared. Copy the https://*.trycloudflare.com line into
#    frontend/.env.local as NEXT_PUBLIC_API_BASE_URL, then `npm run dev`.
import os, re, subprocess, time
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget','-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','+x','/usr/local/bin/cloudflared'], check=True)
tlog = open('/tmp/aiavatar/tunnel.log','w')
tunnel = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://localhost:7860'],
    stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('/tmp/aiavatar/tunnel.log').read())
    if m:
        url = m.group(0); break
print('PUBLIC API URL:', url or 'not found — check tunnel.log')

## Use it from the website
1. `frontend/.env.local` → `NEXT_PUBLIC_API_BASE_URL=<the trycloudflare URL>`
2. `cd frontend && npm run dev`
3. Sign up → **Create Avatar** → upload a 20-30 s video of yourself → wait for *ready*.
4. **Generate** → type a script → download the MP4 (your face + voice, lip-synced).

Logs: `/tmp/aiavatar/{api,worker,tunnel}.log`. Tune `MUSETALK_BBOX_SHIFT` (env var) if the mouth needs alignment. Keep this kernel running while you use the site.